# ⚡ SmartEV - EV Charging Station Queue & Wait Time Prediction
### Research Component: Intelligent EV Charging Optimization & Range Prediction
**Student ID**: IT22134080  
**Platform**: Google Colab / Local Jupyter Notebook  

This notebook contains the complete 16-step machine learning development pipeline:
1. Library Imports
2. Dataset Loading
3. Dataset Inspection
4. Data Cleaning & Type Casting
5. Time-Based Preprocessing
6. Missing Value & Outlier Handling
7. Exploratory Data Analysis (EDA - Diurnal Congestion Patterns)
8. Feature Engineering (Peak Hour, Stall Ratio, Day Type)
9. Feature Selection
10. Train/Test Split
11. Baseline Queuing / Linear Model Training
12. Advanced Multi-Output Random Forest / Gradient Boosting Training
13. Evaluation Metrics (MAE, RMSE)
14. Model Comparison & Visualization
15. Champion Model Selection
16. Model Export (`joblib`) for Drop-in Inference

## Step 1: Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

%matplotlib inline
print("Libraries imported successfully.")

## Step 2 & 3: Load & Inspect Queue Dataset

In [ ]:
csv_path = "../datasets/raw/ev_station_queue_sample.csv"
if not os.path.exists(csv_path):
    csv_path = "ev_station_queue_sample.csv"

df = pd.read_csv(csv_path)
print(f"Dataset shape: {df.shape}")
df.head()

## Step 4 & 5: Time-Based Preprocessing & Cleaning

In [ ]:
# Diurnal peak hours (morning 8-10, evening 17-20)
df['is_peak_hour'] = df['arrival_hour'].apply(lambda h: 1 if h in [8, 9, 10, 17, 18, 19, 20] else 0)
df['is_weekend'] = df['day_of_week'].apply(lambda d: 1 if d in [5, 6] else 0)
df['occupancy_ratio'] = df['currently_occupied'] / df['total_chargers'].clip(lower=1)
df.head()

## Step 6 & 7: Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=df, x='arrival_hour', y='wait_minutes', color='teal')
plt.title('Hourly Average Charging Wait Time (Minutes)')
plt.xlabel('Arrival Hour (0-23)')
plt.ylabel('Average Wait (min)')
plt.show()

## Step 8 & 9: Feature Engineering & Selection

In [ ]:
feature_cols = [
    'total_chargers',
    'currently_occupied',
    'arrival_hour',
    'day_of_week',
    'charging_speed_kw',
    'avg_session_minutes'
]
target_cols = ['queue_length', 'wait_minutes']

X = df[feature_cols]
y = df[target_cols]

## Step 10: Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"Training set: {X_train.shape[0]} samples, Testing set: {X_test.shape[0]} samples")

## Step 11 & 12: Model Training

In [ ]:
# Multi-Output Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
preds = rf_model.predict(X_test)

mae_q = mean_absolute_error(y_test['queue_length'], preds[:, 0])
rmse_q = np.sqrt(mean_squared_error(y_test['queue_length'], preds[:, 0]))

mae_w = mean_absolute_error(y_test['wait_minutes'], preds[:, 1])
rmse_w = np.sqrt(mean_squared_error(y_test['wait_minutes'], preds[:, 1]))

print(f"Random Forest Queue Length -> MAE: {mae_q:.2f} cars | RMSE: {rmse_q:.2f} cars")
print(f"Random Forest Wait Time    -> MAE: {mae_w:.2f} mins | RMSE: {rmse_w:.2f} mins")

## Step 13 & 14: Evaluation & Model Benchmarks

In [ ]:
eval_results = pd.DataFrame([
    {"Target": "Queue Length (cars)", "MAE": round(mae_q, 2), "RMSE": round(rmse_q, 2)},
    {"Target": "Wait Time (minutes)", "MAE": round(mae_w, 2), "RMSE": round(rmse_w, 2)}
])
eval_results

## Step 15 & 16: Champion Model Selection & Drop-in Export

In [ ]:
export_path = "ev_queue_model.joblib"
joblib.dump(rf_model, export_path)
print(f"Exported Queue Model to {export_path}.")
print("\n--- Drop-in Instructions ---")
print("1. Download 'ev_queue_model.joblib'")
print("2. Copy into 'ml-service/saved_models/queue/' in your project.")
print("3. Restart the ML microservice to automatically use this trained model!")